# Gradient Analysis Experiments

This notebook analyzes gradient flow in neural networks with different initialization strategies.

**Experiments:**
1. Single architecture gradient analysis
2. Comparison across initialization strategies
3. Deep network (100+ layers) analysis
4. Zero gradient and dead neuron statistics

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from rp_study.config import ExperimentConfig, NetworkConfig, GradientExperimentConfig
from rp_study.experiments import GradientExperiment, ExperimentResults
from rp_study.experiments.gradient_analysis import compare_initializations
from rp_study.models import FeedForward, list_initializers
from rp_study.models.networks import create_deep_network
from rp_study.visualization.gradient_plots import (
    plot_gradient_histograms,
    plot_row_norm_histograms,
    plot_activation_histograms,
    plot_zero_gradient_stats,
    plot_row_norm_per_layer,
    plot_activation_zero_stats,
    compare_initializations_plot
)

print("Available initialization strategies:")
for init in list_initializers():
    print(f"  - {init}")

## Configuration

Modify these parameters to customize the experiments:

In [ ]:
# ====== EXPERIMENT CONFIGURATION ======

# Base configuration
SEED = 42
DATA_DIR = "../data"

# Dataset configuration
DATASET = "fashion_mnist"  # Options: "mnist", "fashion_mnist"
NUM_SAMPLES = 1000  # Number of samples for gradient computation

# Network architecture
# Format: [input_dim, hidden1, hidden2, ..., output_dim]
LAYER_SIZES = [784, 784, 512, 256, 1]

# Initialization strategy
# Options: "he", "row_centered_he", "custom_variance", "xavier", "uniform_he", "orthogonal"
INIT_STRATEGY = "he"

# For custom_variance initialization
WEIGHT_VARIANCE = None  # Set to a float like 4.0/784 for custom variance

# Histogram bins
NUM_BINS = 50

## 1. Single Architecture Analysis

Run gradient analysis on a single network configuration.

In [ ]:
# Create configurations
exp_config = ExperimentConfig(seed=SEED, data_dir=DATA_DIR)
network_config = NetworkConfig(
    layer_sizes=LAYER_SIZES,
    init_strategy=INIT_STRATEGY,
    weight_variance=WEIGHT_VARIANCE
)
grad_config = GradientExperimentConfig(
    num_samples=NUM_SAMPLES,
    dataset=DATASET,
    num_bins=NUM_BINS
)

print(f"Network architecture: {LAYER_SIZES}")
print(f"Initialization: {INIT_STRATEGY}")
print(f"Dataset: {DATASET} ({NUM_SAMPLES} samples)")

In [ ]:
# Run experiment
experiment = GradientExperiment(exp_config, network_config, grad_config)
results = experiment.run()

print(f"\nExperiment Results:")
print(f"  Loss value: {results.loss_value:.4e}")
print(f"  Device: {results.metadata['device']}")
print(f"\nGradient Statistics:")
for layer, stats in results.grad_stats.items():
    print(f"  {layer}: {stats.zero_proportion:.1%} zeros, mean row norm: {stats.mean_row_norm:.4e}")

In [ ]:
# Plot gradient entry histograms
fig = plot_gradient_histograms(
    results, bins=NUM_BINS,
    title=f"Gradient Entry Distributions ({INIT_STRATEGY} init)"
)
plt.show()

In [ ]:
# Plot row norm histograms
fig = plot_row_norm_histograms(
    results, bins=NUM_BINS,
    title=f"Gradient Row Norms ({INIT_STRATEGY} init)"
)
plt.show()

In [ ]:
# Plot activation histograms
fig = plot_activation_histograms(
    results, bins=NUM_BINS,
    title=f"Activation Distributions ({INIT_STRATEGY} init)"
)
plt.show()

In [ ]:
# Plot zero gradient statistics
fig = plot_zero_gradient_stats(results)
plt.suptitle(f"Zero Gradient Analysis ({INIT_STRATEGY} init)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Plot mean row norm per layer
fig = plot_row_norm_per_layer(results)
plt.title(f"Mean Gradient Row Norm per Layer ({INIT_STRATEGY} init)")
plt.show()

In [ ]:
# Plot activation zero statistics
fig = plot_activation_zero_stats(results)
plt.suptitle(f"Activation Zero Analysis ({INIT_STRATEGY} init)", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Compare Initialization Strategies

Run the same experiment with different initialization strategies.

**Important distinction:**
- **Gradient entry zeros**: Individual entries in ∂L/∂W that are exactly 0
- **Activation zeros**: Post-ReLU activations that are 0 (~50% due to ReLU's nature)
- **Truly inactive neurons**: Neurons that output 0 for ALL samples (neuron death)

In deep networks with standard He init, ~50% of gradient entries become zero because 
dead neurons produce entirely-zero gradient rows. Row-centered He may prevent this.

In [ ]:
# Strategies to compare
STRATEGIES_TO_COMPARE = ["he", "row_centered_he", "xavier", "uniform_he"]

# Run comparison
comparison_results = compare_initializations(
    layer_sizes=LAYER_SIZES,
    init_strategies=STRATEGIES_TO_COMPARE,
    num_samples=NUM_SAMPLES,
    dataset=DATASET,
    seed=SEED
)

# Print detailed comparison results
print("=" * 60)
print("COMPARISON RESULTS")
print("=" * 60)

for strategy, result in comparison_results.items():
    # Gradient entry zeros (dead neurons cause entire rows to be zero)
    grad_zero_props = result.get_zero_gradient_proportions()
    avg_grad_zero = np.mean(list(grad_zero_props.values()))
    
    # Activation zeros (ReLU zeros out negative pre-activations)
    act_zero_props = result.get_activation_zero_proportions()
    avg_act_zero = np.mean(list(act_zero_props.values()))
    
    # Truly inactive neurons (zero for ALL samples = dead neurons)
    inactive_props = result.get_truly_inactive_proportions()
    avg_inactive = np.mean(list(inactive_props.values()))
    
    print(f"\n{strategy.upper()}:")
    print(f"  Gradient entry zeros:   {avg_grad_zero:.1%}  (entire rows become 0 when neurons die)")
    print(f"  Activation zeros:       {avg_act_zero:.1%}  (expected ~50% due to ReLU)")
    print(f"  Truly inactive neurons: {avg_inactive:.1%}  (zero for ALL samples = dead)")

print("\n" + "=" * 60)

In [ ]:
# Compare zero gradient proportions (excluding output layer for better visualization)
fig = compare_initializations_plot(
    comparison_results,
    metric="zero_proportion",
    exclude_output_layer=True  # Output layer has much larger gradients
)
plt.show()

In [ ]:
# Compare mean row norms (excluding output layer which dominates y-axis)
fig = compare_initializations_plot(
    comparison_results,
    metric="mean_row_norm",
    exclude_output_layer=True
)
plt.show()

# Also show with log scale for better comparison
fig = compare_initializations_plot(
    comparison_results,
    metric="mean_row_norm",
    exclude_output_layer=True,
    use_log_scale=True
)
plt.title("Mean Gradient Row Norm (log scale, hidden layers only)")
plt.show()

## 3. Deep Network Analysis (100+ layers)

Analyze gradient flow in very deep networks.

In [ ]:
# Deep network configuration
N_HIDDEN_LAYERS = 100
WIDTH_RANGE = (100, 1000)  # Random widths between 100 and 1000

# Create deep network
deep_net = create_deep_network(
    input_dim=784,
    output_dim=1,
    num_hidden_layers=N_HIDDEN_LAYERS,
    hidden_widths="random",
    width_range=WIDTH_RANGE,
    init_strategy="he",
    seed=SEED
)

print(f"Deep network created:")
print(f"  Number of layers: {deep_net.num_layers}")
print(f"  Architecture: {deep_net.layer_sizes[:5]}...{deep_net.layer_sizes[-3:]}")
print(f"  Total parameters: {sum(p.numel() for p in deep_net.parameters()):,}")

In [ ]:
# Run deep network experiment
deep_network_config = NetworkConfig(
    layer_sizes=deep_net.layer_sizes,
    init_strategy="he"
)

deep_experiment = GradientExperiment(exp_config, deep_network_config, grad_config)
deep_results = deep_experiment.run()

print(f"\nDeep Network Results:")
print(f"  Loss: {deep_results.loss_value:.4e}")

In [ ]:
# Plot mean row norm across all layers
fig = plot_row_norm_per_layer(deep_results, figsize=(15, 5))
plt.title(f"Gradient Row Norm across {N_HIDDEN_LAYERS} Hidden Layers")
plt.xticks(rotation=90, fontsize=6)
plt.show()

In [ ]:
# Plot zero gradient statistics
fig = plot_zero_gradient_stats(deep_results, figsize=(15, 5))
for ax in fig.axes:
    ax.tick_params(axis='x', rotation=90, labelsize=6)
plt.suptitle(f"Zero Gradient Analysis ({N_HIDDEN_LAYERS} hidden layers)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Custom Variance Experiments

Compare different variance scales (2/d, 2.5/d, 4/d).

In [ ]:
# Variance scales to test
VARIANCE_SCALES = [2.0, 2.5, 3.0, 4.0]  # multiplier for 1/d
d = LAYER_SIZES[0]  # input dimension

variance_results = {}

for scale in VARIANCE_SCALES:
    variance = scale / d
    exp_config.setup_seeds()  # Reset for fair comparison
    
    net_config = NetworkConfig(
        layer_sizes=LAYER_SIZES,
        init_strategy="custom_variance",
        weight_variance=variance
    )
    
    experiment = GradientExperiment(exp_config, net_config, grad_config)
    result = experiment.run()
    variance_results[f"{scale}/d"] = result
    
    print(f"Variance {scale}/d: loss = {result.loss_value:.4e}")

In [ ]:
# Compare variance scales
fig = compare_initializations_plot(
    variance_results,
    metric="zero_proportion"
)
plt.title("Zero Gradient Proportion by Variance Scale")
plt.show()

In [ ]:
fig = compare_initializations_plot(
    variance_results,
    metric="mean_row_norm"
)
plt.title("Mean Gradient Row Norm by Variance Scale")
plt.show()

## Summary

Key findings from this analysis:

1. **Gradient sparsity**: Proportion of zero gradients varies by initialization
2. **Row norms**: How gradient magnitude distributes across layers
3. **Deep networks**: Gradient flow characteristics in 100+ layer networks
4. **Variance scaling**: Effect of different variance scales on gradient statistics